# 04c — Scaling con RobustScaler

Scala la feature matrix con `RobustScaler` (mediana=0, scala=IQR).
Robusto agli outlier residui che possono sopravvivere al log1p.

Produce:
- `data/feature_matrix_scaled_v1.parquet`
- `data/scaler_v1.joblib`

**Prerequisito**: `data/feature_matrix_graph_v1.parquet` esiste (output di 04b).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

df = pd.read_parquet("../data/feature_matrix_graph_v1.parquet")
print(f"Caricato: {df.shape}")
print(f"NaN: {df.isnull().sum().sum()} (atteso: 0)")
print(f"Colonne: {list(df.columns)}")

Caricato: (9096, 20)
NaN: 0 (atteso: 0)
Colonne: ['agent_id', 'n_posts', 'n_comments', 'n_comments_received', 'active_days', 'burstiness_posts', 'hour_entropy', 'reply_to_post_ratio', 'self_reply_rate', 'mean_thread_depth', 'mean_post_length', 'std_post_length', 'type_token_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'local_clustering', 'egonet_density', 'reciprocity_local']


In [2]:
# Separa agent_id dalle feature
feature_cols = [c for c in df.columns if c != "agent_id"]
print(f"Feature da scalare ({len(feature_cols)}): {feature_cols}")

Feature da scalare (19): ['n_posts', 'n_comments', 'n_comments_received', 'active_days', 'burstiness_posts', 'hour_entropy', 'reply_to_post_ratio', 'self_reply_rate', 'mean_thread_depth', 'mean_post_length', 'std_post_length', 'type_token_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'local_clustering', 'egonet_density', 'reciprocity_local']


In [3]:
# Fitting e trasformazione
scaler = RobustScaler()
scaled_values = scaler.fit_transform(df[feature_cols])

df_scaled = pd.DataFrame(scaled_values, columns=feature_cols)
df_scaled.insert(0, "agent_id", df["agent_id"].values)

print(f"Shape post-scaling: {df_scaled.shape}")
print(f"NaN: {df_scaled.isnull().sum().sum()} (atteso: 0)")

Shape post-scaling: (9096, 20)
NaN: 0 (atteso: 0)


In [4]:
# Verifica: per RobustScaler, la mediana di ogni feature scalata dovrebbe essere ~0
# e l'IQR ~1 (per definizione)
print("Statistiche post-scaling (campione di feature):")
stats = df_scaled[feature_cols].agg(["median", lambda x: x.quantile(0.75) - x.quantile(0.25)])
stats.index = ["median", "IQR"]
print(stats.round(4).to_string())

Statistiche post-scaling (campione di feature):
        n_posts  n_comments  n_comments_received  active_days  burstiness_posts  hour_entropy  reply_to_post_ratio  self_reply_rate  mean_thread_depth  mean_post_length  std_post_length  type_token_ratio  in_degree  out_degree  pagerank  betweenness  local_clustering  egonet_density  reciprocity_local
median      0.0         0.0                  0.0          0.0               0.0           0.0                  0.0              0.0                0.0               0.0              0.0               0.0        0.0         0.0       0.0          0.0               0.0             0.0                0.0
IQR         1.0         1.0                  1.0          1.0               1.0           1.0                  1.0              0.0                1.0               1.0              1.0               1.0        1.0         1.0       1.0          1.0               1.0             1.0                1.0


In [5]:
# Statistiche descrittive complete
print("Describe post-scaling:")
print(df_scaled[feature_cols].describe().round(4).to_string())

Describe post-scaling:
         n_posts  n_comments  n_comments_received  active_days  burstiness_posts  hour_entropy  reply_to_post_ratio  self_reply_rate  mean_thread_depth  mean_post_length  std_post_length  type_token_ratio  in_degree  out_degree   pagerank  betweenness  local_clustering  egonet_density  reciprocity_local
count  9096.0000   9096.0000            9096.0000    9096.0000         9096.0000     9096.0000            9096.0000        9096.0000          9096.0000         9096.0000        9096.0000         9096.0000  9096.0000   9096.0000  9096.0000    9096.0000         9096.0000       9096.0000          9096.0000
mean      0.1654      0.1481               0.2918       0.4187            0.6981       -0.0055              -0.1691           0.0140             0.3701           -0.2174          -0.1509            0.0110     0.1274      0.1595     0.3980       1.3242            0.7625          0.3742             0.7951
std       0.6663      0.7365               0.8688       1.0962

In [6]:
# Salvataggio parquet
parquet_path = "../data/feature_matrix_scaled_v1.parquet"
df_scaled.to_parquet(parquet_path, index=False)
print(f"Salvato: {parquet_path}")

# Salvataggio scaler
scaler_path = "../data/scaler_v1.joblib"
joblib.dump(scaler, scaler_path)
print(f"Salvato: {scaler_path}")

Salvato: ../data/feature_matrix_scaled_v1.parquet
Salvato: ../data/scaler_v1.joblib


In [7]:
# Verifica idempotenza: ricarica e controlla shape
df_reload = pd.read_parquet(parquet_path)
scaler_reload = joblib.load(scaler_path)

assert df_reload.shape == df_scaled.shape, "Shape mismatch al ricaricamento parquet!"
assert hasattr(scaler_reload, "center_"), "Scaler non correttamente serializzato!"

print(f"Verifica parquet: OK — shape {df_reload.shape}")
print(f"Verifica scaler: OK — center_ shape {scaler_reload.center_.shape}")
print("\nDone.")

Verifica parquet: OK — shape (9096, 20)
Verifica scaler: OK — center_ shape (19,)

Done.
